In [43]:
from ftplib import FTP
import pandas as pd
from pathlib import Path
import unidecode
import numpy as np

# !pip install py7zr
import py7zr
from pathlib import Path
from ftplib import FTP

ARQUIVO = "RAIS_ESTAB_PUB.7z"
PATH = Path("/lakehouse/default/Files/rais_ftp")

def download_rais_data():
    # Acessar FTP
    ftp = FTP("ftp.mtps.gov.br", encoding="latin1")
    ftp.login()
    ftp.cwd(f"pdet/microdados/RAIS/2024")

    # Baixar arquivo zipado
    with open(PATH / ARQUIVO, 'wb') as file:
        ftp.retrbinary(f"RETR {ARQUIVO}", file.write)

    # Extrair arquivo TXT
    with py7zr.SevenZipFile(PATH / ARQUIVO, mode='r') as archive:
        archive.extractall(path=PATH)


def processar_rais_ftp():
    rais_ftp = pd.read_csv(
        PATH / "RAIS_ESTAB_PUB.COMT",
        # sep=";",
        encoding="latin1",
        dtype={"Município": str, "CNAE 2.0 Subclasse": str, "UF": str},
    )
    return rais_ftp


def tratar_nome_colunas_ftp(rais_ftp):
    rais_ftp.columns = [unidecode.unidecode(col) for col in rais_ftp.columns]
    rais_ftp.columns = (
        rais_ftp.columns
        .str.lower()
        .str.replace("-", "")
        .str.strip()
        .str.replace("  ", " ")
        .str.replace(" ", "_")
        .str.replace(".", "")
    )
    return rais_ftp


def ajustar_base_ftp_padrao(rais_ftp):
    rais_ftp['ano'] = 2024

    rais_ftp = rais_ftp.loc[rais_ftp['uf_codigo'] == 35].copy()
    rais_ftp['sigla_uf'] = 'SP'

    rais_ftp = rais_ftp.rename(columns={
        "municipio_codigo":"id_municipio",
        "qtd_vinculos_ativos":"quantidade_vinculos_ativos",
        "qtd_vinculos_clt": "quantidade_vinculos_clt",
        "qtd_vinculos_estatutarios": "quantidade_vinculos_estatutarios",
        "cnae_20_classe_codigo": "cnae_2",
        "cnae_20_subclasse_codigo": "cnae_2_subclasse",
        "tamanho_estabelecimento_codigo": "tamanho_estabelecimento"
    })

    rais_ftp_grp = rais_ftp.groupby([
        "ano", "sigla_uf", "id_municipio",
        "tamanho_estabelecimento", "cnae_2",
        "cnae_2_subclasse"
    ], as_index=False).agg({
        "quantidade_vinculos_ativos": "sum",
        "quantidade_vinculos_clt": "sum",
        "quantidade_vinculos_estatutarios": "sum"
    })
    rais_ftp_grp['cnae_1'] = np.nan
    return rais_ftp_grp


def main_rais_ftp():
    # download_rais_data()
    rais_ftp = processar_rais_ftp()
    rais_ftp_colunas = tratar_nome_colunas_ftp(rais_ftp)
    rais_ftp_colunas_padrao = ajustar_base_ftp_padrao(rais_ftp_colunas)
    return rais_ftp_colunas_padrao


def consolidar_rais():

    rais_dump = spark.sql("SELECT * FROM lh_cidade_inteligente_osasco.raw_rais_estab_sp").toPandas()
    rais_ftp = main_rais_ftp()
    rais = pd.concat([rais_dump, rais_ftp], axis=0).reset_index(drop=True)

    return rais


rais = consolidar_rais()

StatementMeta(, 94ec728d-37d7-474a-bed3-5877ca9ac780, 45, Finished, Available, Finished)

In [45]:
spark_df = spark.createDataFrame(rais)
(
    spark_df
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema","true")
    .saveAsTable("raw_rais_estab_sp")
)

StatementMeta(, 94ec728d-37d7-474a-bed3-5877ca9ac780, 47, Finished, Available, Finished)

/opt/spark/python/lib/pyspark.zip/pyspark/sql/pandas/conversion.py:351: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Expected bytes, got a 'int' object
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


In [31]:
from ftplib import FTP

ftp = FTP("ftp.mtps.gov.br", encoding="latin1")
ftp.login()
ftp.cwd(f"pdet/microdados/RAIS/2024 Parcial")
ftp.nlst()

StatementMeta(, 94ec728d-37d7-474a-bed3-5877ca9ac780, 33, Finished, Available, Finished)

['RAIS_ESTAB_PUB.7z',
 'RAIS_VINC_PUB_CENTRO_OESTE.7z',
 'RAIS_VINC_PUB_MG_ES_RJ.7z',
 'RAIS_VINC_PUB_NI.7z',
 'RAIS_VINC_PUB_NORDESTE.7z',
 'RAIS_VINC_PUB_NORTE.7z',
 'RAIS_VINC_PUB_SP.7z',
 'RAIS_VINC_PUB_SUL.7z']